# QLoRA baseline (RunPod) - 3000 train+val split

`QLoRA_baseline.py` 흐름을 따르되 아래 변경사항을 반영합니다.

- 학습 샘플: 2000 -> **3000**
- 데이터 셔플: **shuffle(seed=42)**
- 검증 데이터: **랜덤 150개 (5%)**
- 학습 중 검증 loss 모니터링(`eval_dataset`)
- `warmup_ratio=0.05`
- 모델/데이터는 HF에서 자동 로드


In [ ]:
# ============================================================
# 0) 공통 import + 실험 설정값 모음
# ------------------------------------------------------------
# 이 셀에서 실험의 핵심 하이퍼파라미터를 한 번에 관리합니다.
# (학습/평가/양자화/저장 경로)
# ============================================================
import os
import gc
import zipfile
from pathlib import Path
from typing import List, Optional

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    default_data_collator,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

# ---------------------------
# RunPod/환경 변수 기반 기본 경로
# ---------------------------
WORKSPACE = os.environ.get("WORKSPACE", "/workspace")
MODEL_ID = os.environ.get("MODEL_ID", "LGAI-EXAONE/EXAONE-4.0-1.2B")
DATASET_ID = os.environ.get("DATASET_ID", "LGAI-EXAONE/MANTA-1M")
DATASET_SPLIT = "train"

# ---------------------------
# 데이터 분할 규칙
# ---------------------------
SEED = 42
NUM_TOTAL_SAMPLES = 3000          # shuffle 후 앞에서 3000개 사용
NUM_EVAL_SAMPLES = 150            # 3000의 5%를 검증셋으로 사용

# ---------------------------
# 학습 시퀀스/전처리 규칙
# ---------------------------
MAX_TRAIN_SEQ_LEN = 1024
MIN_PROMPT_TOKENS = 128

# ---------------------------
# 학습 하이퍼파라미터
# ---------------------------
TRAIN_BATCH_SIZE = 1
GRAD_ACC = 16
LEARNING_RATE = 1e-5
NUM_TRAIN_EPOCHS = 1
WARMUP_RATIO = 0.05               # 요청사항 반영: warmup ratio 0.05
EVAL_STEPS = 20                   # 학습 중 val loss 로깅 주기

# ---------------------------
# LoRA 설정
# ---------------------------
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGETS = ["auto"]          # 모델 구조를 보고 target suffix 자동 추출

# ---------------------------
# GPTQ 설정
# ---------------------------
NUM_CALIBRATION_SAMPLES = 1024
MAX_CALIB_SEQ_LEN = 1024
IGNORE_MODE = "base"
DEEP_PROTECT_RATIO = 0.0
GPTQ_DAMPENING_FRAC = 0.001
GPTQ_BLOCK_SIZE = 128

# baseline.py 기본 동작과 동일하게 저장 압축은 기본 OFF
SAVE_COMPRESSED = False
MERGE_DTYPE = "fp16"

# ---------------------------
# 출력 경로
# ---------------------------
WORKSPACE = str(Path(WORKSPACE).resolve())
ADAPTER_DIR = str(Path(WORKSPACE) / "qlora_adapter_3000_val")
OUT_DIR = str(Path(WORKSPACE) / "out" / "model_w8a8_3000_val")
OUT_ZIP = str(Path(WORKSPACE) / "submit_3000_val.zip")

print(f"[INFO] WORKSPACE={WORKSPACE}")
print(f"[INFO] MODEL_ID={MODEL_ID}")
print(f"[INFO] DATASET_ID={DATASET_ID}")


In [ ]:
# ============================================================
# 1) 유틸 함수
# ------------------------------------------------------------
# - 경로 생성
# - GPU/BF16 지원 여부
# - HF 캐시 경로 고정(RunPod 재실행 시 다운로드 재사용)
# - LoRA target 자동 탐지
# - GPTQ 저장 전 quantization format 정리(필요 시)
# ============================================================
def ensure_dir(path) -> str:
    """경로가 없으면 생성하고 문자열 경로를 반환."""
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return str(p)

def cuda_available() -> bool:
    """CUDA 사용 가능 여부."""
    return torch.cuda.is_available()

def bf16_supported() -> bool:
    """현재 GPU에서 BF16을 안정적으로 쓸 수 있는지 확인."""
    return cuda_available() and getattr(torch.cuda, "is_bf16_supported", lambda: False)()

def pick_compute_dtype():
    """4bit 학습 시 내부 계산 dtype 선택 (BF16 > FP16 > FP32)."""
    if bf16_supported():
        return torch.bfloat16
    return torch.float16 if cuda_available() else torch.float32

def setup_hf_cache(workspace: str) -> None:
    """HF 캐시를 workspace 아래로 고정해 재시작/재학습 시 재사용."""
    cache_root = Path(workspace) / ".cache" / "huggingface"
    os.environ.setdefault("HF_HOME", str(cache_root))
    os.environ.setdefault("HF_DATASETS_CACHE", str(cache_root / "datasets"))
    os.environ.setdefault("TRANSFORMERS_CACHE", str(cache_root / "transformers"))
    ensure_dir(cache_root)
    ensure_dir(cache_root / "datasets")
    ensure_dir(cache_root / "transformers")

def infer_lora_targets_from_model(model, preferred_suffixes):
    """
    모델 내부 Linear 모듈 이름을 스캔해서,
    preferred suffix(q_proj, k_proj ...) 중 실제 존재하는 suffix만 반환.
    """
    candidates = []
    for name, module in model.named_modules():
        if not isinstance(module, torch.nn.Linear):
            continue
        suffix = name.split(".")[-1]
        if suffix in preferred_suffixes:
            candidates.append(name)

    suffix_hits = []
    for s in preferred_suffixes:
        if any(n.split(".")[-1] == s for n in candidates):
            suffix_hits.append(s)

    if not suffix_hits:
        raise RuntimeError("LoRA target 자동 탐지 실패")
    return suffix_hits

def get_deep_ignore_patterns(model: torch.nn.Module, protect_ratio: float = 0.0) -> List[str]:
    """
    GPTQ에서 마지막 N% 레이어를 보호(양자화 제외)할 때 사용할 ignore prefix 생성.
    protect_ratio=0.0이면 보호 안 함.
    """
    if protect_ratio <= 0:
        return []

    total_layers = 0
    layer_prefix = None
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        total_layers, layer_prefix = len(model.model.layers), "model.layers"
    elif hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        total_layers, layer_prefix = len(model.transformer.h), "transformer.h"

    if total_layers == 0 or layer_prefix is None:
        return []

    num_protect = max(1, int(total_layers * protect_ratio))
    start_idx = total_layers - num_protect
    return [f"{layer_prefix}.{i}." for i in range(start_idx, total_layers)]

def _fix_quantization_format_warning(model: torch.nn.Module, save_compressed: bool) -> None:
    """
    save_compressed=True일 때만 quantization format 메타를 재정리.
    False면 baseline.py와 동일하게 즉시 return(no-op).
    """
    if not save_compressed:
        return

    # 기존 format 값이 박혀 있으면 충돌 경고가 날 수 있어 우선 초기화
    for m in model.modules():
        qs = getattr(m, "quantization_scheme", None)
        if qs is None:
            continue
        if hasattr(qs, "format") and getattr(qs, "format") is not None:
            setattr(qs, "format", None)

    # 버전에 따라 helper 경로가 없을 수 있으므로 예외는 경고만 출력
    try:
        from llmcompressor.transformers.compression.quantization_format import infer_and_set_per_module_quantization_format
        infer_and_set_per_module_quantization_format(model, quantization_format=None, save_compressed=True, sparsity_structure=None)
    except Exception as e:
        print(f"[WARN] quantization_format 재추론 스킵 ({type(e).__name__}: {e})")

# 캐시/출력 경로 준비
setup_hf_cache(WORKSPACE)
ensure_dir(ADAPTER_DIR)
ensure_dir(OUT_DIR)


In [ ]:
# ============================================================
# 2) 모델/토크나이저 로드 + 데이터 3000 샘플 구성
# ------------------------------------------------------------
# 순서:
# 1) 토크나이저 준비
# 2) 4bit QLoRA 학습용 모델 로드
# 3) 데이터셋 전체 로드 -> shuffle(seed=42) -> 3000개 절단
# 4) 랜덤 150개를 eval로 분리(5%)
# ============================================================
compute_dtype = pick_compute_dtype()

# tokenizer 준비 (pad token 없으면 eos로 대체)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
eos_id = tokenizer.eos_token_id

# QLoRA 4bit 설정 (NF4 + double quant)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

print("[INFO] 모델 로드(4bit NF4 + DQ)")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto" if cuda_available() else None,
    trust_remote_code=True,
)

# k-bit 학습 준비 + gradient checkpointing
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
# 학습 중에는 use_cache 비활성화가 일반적
model.config.use_cache = False

print(f"[INFO] 데이터셋 로드: {DATASET_ID}")
raw = load_dataset(DATASET_ID, split=DATASET_SPLIT, trust_remote_code=True)

# 요청사항: 반드시 seed=42로 셔플 후 샘플 추출
raw = raw.shuffle(seed=SEED)
raw3000 = raw.select(range(min(NUM_TOTAL_SAMPLES, len(raw))))

# 요청사항: 3000 중 랜덤 150개를 eval로 분리
splits = raw3000.train_test_split(test_size=NUM_EVAL_SAMPLES, seed=SEED, shuffle=True)
train_raw = splits["train"]
eval_raw = splits["test"]
print(f"[INFO] train={len(train_raw)}, eval={len(eval_raw)}")


In [ ]:
# ============================================================
# 3) 전처리: SFT 학습 입력/라벨 생성
# ------------------------------------------------------------
# 규칙:
# - conversations 마지막 turn이 assistant인 샘플만 사용
# - prompt는 loss 제외(-100), answer 토큰만 loss 계산
# - max_seq_len(1024) 초과 시 baseline.py 방식으로 절단
# ============================================================
def preprocess_train(example):
    convs = example.get("conversations", None)
    if not convs or not isinstance(convs, list):
        return {"input_ids": [], "attention_mask": [], "labels": []}

    # SFT 포맷: 마지막이 assistant 응답이어야 정답(label) 생성 가능
    if convs[-1].get("role") != "assistant":
        return {"input_ids": [], "attention_mask": [], "labels": []}

    # prompt: assistant 마지막 턴 전까지, answer: 마지막 assistant 내용
    prompt_convs = convs[:-1]
    answer_text = convs[-1].get("content", "") or ""

    # prompt는 chat_template 적용, answer는 raw tokenize
    prompt_text = tokenizer.apply_chat_template(prompt_convs, add_generation_prompt=True, tokenize=False)
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(answer_text, add_special_tokens=False)["input_ids"]

    # answer 끝에 eos 보장
    if not answer_ids or answer_ids[-1] != eos_id:
        answer_ids = answer_ids + [eos_id]

    # 길이 제한: baseline.py와 동일한 절단 정책
    total = len(prompt_ids) + len(answer_ids)
    max_len = MAX_TRAIN_SEQ_LEN
    if total > max_len:
        # 프롬프트는 뒤쪽 정보가 더 중요하다고 보고 tail 유지
        keep_prompt = min(len(prompt_ids), max_len)
        keep_prompt = max(keep_prompt, MIN_PROMPT_TOKENS)
        keep_prompt = min(keep_prompt, len(prompt_ids))
        prompt_ids = prompt_ids[-keep_prompt:]

        # 남은 budget 내에서 answer 유지
        budget = max_len - len(prompt_ids)
        if budget <= 0:
            prompt_ids = prompt_ids[-(max_len - 1):]
            answer_ids = [eos_id]
        else:
            if len(answer_ids) > budget:
                answer_ids = answer_ids[: max(1, budget - 1)] + [eos_id]

    # 최종 입력
    input_ids = prompt_ids + answer_ids
    attention_mask = [1] * len(input_ids)

    # prompt 구간은 loss 제외, answer 구간만 학습
    labels = [-100] * len(prompt_ids) + answer_ids

    # 고정 길이 패딩(배치 텐서화 안정화)
    pad_id = tokenizer.pad_token_id or eos_id
    pad_len = max_len - len(input_ids)
    if pad_len > 0:
        input_ids += [pad_id] * pad_len
        attention_mask += [0] * pad_len
        labels += [-100] * pad_len

    # 안전 장치: 마지막 토큰 eos 보정
    input_ids[-1] = eos_id
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

# train/eval 각각 동일 전처리 적용
train_ds = train_raw.map(preprocess_train, remove_columns=train_raw.column_names)
train_ds = train_ds.filter(lambda x: len(x["input_ids"]) > 0)

eval_ds = eval_raw.map(preprocess_train, remove_columns=eval_raw.column_names)
eval_ds = eval_ds.filter(lambda x: len(x["input_ids"]) > 0)

print(f"[INFO] 최종 train={len(train_ds)}, eval={len(eval_ds)}")


In [ ]:
# ============================================================
# 4) LoRA 적용 + Trainer 학습
# ------------------------------------------------------------
# 요청사항 반영:
# - warmup_ratio=0.05
# - eval_dataset 사용으로 학습 중 val loss 확인
# - 검증 주기 eval_steps=20
# ============================================================

# auto 모드면 모델에 실제 존재하는 target suffix만 사용
preferred = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"] if LORA_TARGETS == ["auto"] else LORA_TARGETS
lora_targets = infer_lora_targets_from_model(model, preferred)
print(f"[INFO] LoRA target_modules={lora_targets}")

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=lora_targets,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

use_bf16 = bf16_supported()
use_fp16 = cuda_available() and not use_bf16

# 체크포인트 저장 경로
ckpt_dir = ensure_dir(Path(WORKSPACE) / "qlora_ckpt_3000_val")

training_args = TrainingArguments(
    output_dir=str(ckpt_dir),
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    logging_steps=20,

    # validation loss를 학습 중 주기적으로 확인
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,

    save_steps=500,
    save_total_limit=2,
    bf16=use_bf16,
    fp16=use_fp16,
    report_to=[],
    optim="adamw_torch",
    remove_unused_columns=False,

    # 가장 좋은 eval_loss 체크포인트 로드
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=default_data_collator,
)

print("[INFO] QLoRA 학습 시작 (val loss 모니터링)")
trainer.train()
print("[INFO] QLoRA 학습 완료")

# ------------------------------------------------------------
# 학습 로그 요약: train loss / validation(eval) loss 모두 확인
# ------------------------------------------------------------
# Trainer는 step마다 log_history를 남깁니다.
# - train loss key: loss
# - val loss key: eval_loss
history = trainer.state.log_history
train_logs = [(x.get("step"), x.get("loss")) for x in history if "loss" in x and "eval_loss" not in x]
eval_logs = [(x.get("step"), x.get("eval_loss")) for x in history if "eval_loss" in x]

if train_logs:
    last_step, last_train_loss = train_logs[-1]
    print(f"[INFO] 마지막 train loss: step={last_step}, loss={last_train_loss:.6f}")
else:
    print("[WARN] train loss 로그를 찾지 못했습니다.")

if eval_logs:
    last_eval_step, last_eval_loss = eval_logs[-1]
    print(f"[INFO] 마지막 eval loss: step={last_eval_step}, eval_loss={last_eval_loss:.6f}")
else:
    print("[WARN] eval loss 로그를 찾지 못했습니다. (eval_steps/eval_dataset 설정 확인)")

# 시각화(선택): matplotlib가 있으면 곡선 표시
try:
    import matplotlib.pyplot as plt

    if train_logs or eval_logs:
        plt.figure(figsize=(8, 4.5))
        if train_logs:
            ts, tl = zip(*train_logs)
            plt.plot(ts, tl, label="train loss", marker="o", linewidth=1)
        if eval_logs:
            es, el = zip(*eval_logs)
            plt.plot(es, el, label="eval loss", marker="s", linewidth=1)
        plt.xlabel("step")
        plt.ylabel("loss")
        plt.title("Training vs Validation Loss")
        plt.grid(True, alpha=0.25)
        plt.legend()
        plt.show()
except Exception as e:
    print(f"[WARN] loss plot 스킵: {e}")

# 학습된 LoRA adapter 저장
ensure_dir(ADAPTER_DIR)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"[INFO] LoRA 어댑터 저장: {ADAPTER_DIR}")


In [ ]:
# ============================================================
# 5) (선택) Merge + GPTQ + zip
# ------------------------------------------------------------
# - Dacon 제출 모델이 필요할 때 실행
# - 학습만 확인하려면 이 셀은 건너뛰어도 됩니다.
# ============================================================

# Merge 단계 dtype 결정
merge_dtype = torch.float16 if MERGE_DTYPE == "fp16" else torch.bfloat16
if not cuda_available():
    merge_dtype = torch.float32

# 베이스 모델 로드 -> LoRA adapter 결합 -> 단일 모델로 merge
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=merge_dtype,
    device_map=None,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
base_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
base_model = base_model.merge_and_unload()
base_model.eval()
base_model.config.use_cache = True
if cuda_available():
    base_model.to("cuda")

# GPTQ에서 보호할 모듈 preset
ignore_presets = {
    "base": ["embed_tokens", "lm_head"],
    "plus_o": ["embed_tokens", "lm_head", "o_proj"],
    "plus_down": ["embed_tokens", "lm_head", "down_proj"],
    "plus_o_down": ["embed_tokens", "lm_head", "o_proj", "down_proj"],
    "attn_all": ["embed_tokens", "lm_head", "q_proj", "k_proj", "v_proj", "o_proj"],
}
base_ignore = ignore_presets.get(IGNORE_MODE, ignore_presets["base"])
deep_patterns = get_deep_ignore_patterns(base_model, protect_ratio=DEEP_PROTECT_RATIO)
final_ignore = list(base_ignore) + list(deep_patterns)

# 캘리브 데이터 준비 (assistant 응답 제거한 prompt만 사용)
calib_raw = load_dataset(DATASET_ID, split=f"{DATASET_SPLIT}[:{NUM_CALIBRATION_SAMPLES}]", trust_remote_code=True)

def to_calib_text(example):
    convs = example.get("conversations", None)
    if not convs:
        return {"text": ""}
    if convs and convs[-1].get("role") == "assistant":
        convs = convs[:-1]
    if not convs:
        return {"text": ""}
    return {"text": tokenizer.apply_chat_template(convs, add_generation_prompt=True, tokenize=False)}

calib_ds = calib_raw.map(to_calib_text, remove_columns=calib_raw.column_names)
calib_ds = calib_ds.filter(lambda x: len(x.get("text", "").strip()) > 0)

# GPTQ 양자화 (W8A8)
print("[INFO] GPTQ W8A8 시작")
recipe = [
    GPTQModifier(
        scheme="W8A8",
        targets=["Linear"],
        ignore=final_ignore,
        actorder="static",
        dampening_frac=GPTQ_DAMPENING_FRAC,
        block_size=GPTQ_BLOCK_SIZE,
        offload_hessians=False,
    )
]
oneshot(
    model=base_model,
    dataset=calib_ds,
    recipe=recipe,
    max_seq_length=MAX_CALIB_SEQ_LEN,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)
print("[INFO] GPTQ 완료")

# save_compressed=True일 때만 format 재정리 로직 동작
_fix_quantization_format_warning(base_model, save_compressed=SAVE_COMPRESSED)

# 모델 저장
save_kwargs = {}
if SAVE_COMPRESSED:
    save_kwargs["save_compressed"] = True
base_model.save_pretrained(OUT_DIR, safe_serialization=True, **save_kwargs)
tokenizer.save_pretrained(OUT_DIR)
print(f"[INFO] 모델 저장 완료: {OUT_DIR}")

# 제출 형식 zip 생성 (model/ 하위 구조)
with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in Path(OUT_DIR).rglob("*"):
        if f.is_file():
            zf.write(f, "model/" + str(f.relative_to(OUT_DIR)).replace("\\", "/"))
print(f"[INFO] 제출용 zip: {OUT_ZIP}")

# 메모리 정리
# (긴 노트북 세션에서 OOM 방지를 위해 명시적으로 해제)
del base_model
gc.collect()
if cuda_available():
    torch.cuda.empty_cache()
